# Anomaly check

QC on satellite GVF vs PhenoCam GCC and NDVI: scores table, gap-by-veg boxplots,
and a golden-standard ranking.

**Spin-up** (`gvf_sos == 1`): the phenology fit failed and landed on DOY 1 by
accident, not because green-up really started on Jan 1. On flat, low amplitude
curves (EN, sparse shrub, evergreen) there is no clear winter to summer swing, so
it pin SOS at the first day of data. That inflates gap /
divergence vs NDVI or GCC (noise misread as signal), so spin-up sites must be
flagged and usually excluded before interpreting lag or compression.

Artifacts: `anomaly_pipeline/output/` (`metadata/` scores, `boxplot/`,
`golden_standard_ranking.csv`).

**Veg Codes** DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub

In [8]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import (
    build_golden_ranking,
    collect_folder,
    group_summary,
    load_table,
    plot_gap_boxplot_by_veg,
    top_n,
)

# GVF text in plotting stage (drop once, reuse here)
INPUT_DIR = REPO / "plotting_pipeline" / "input"
ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"

## MetaData table

One row per site-year: SOS/MOS/DOS/EOS for GVF, GCC, and NDVI, plus pairwise
gap / DTW / divergence. Use it to find spin-up (`gvf_sos == 1`), large land-type
offsets, and other bad fits without opening every plot.

Writes `anomaly_pipeline/output/metadata/<FOLDER>_scores.csv`.


In [9]:
FOLDER = "GBOV_2023"  # GBOV_2024 / GoldenSites_2023
LIMIT = None
SORT = "gvf_vs_ndvi_div"
TOP = 10

csv_path = collect_folder(FOLDER, INPUT_DIR, ANOMALY_DIR, limit=LIMIT)
df = load_table(csv_path)
print(csv_path, "|", len(df), "rows")
df.head()

  BART (DB, 2023): GVF-GCC div=1.75  GVF-NDVI div=1.44  GCC-NDVI div=1.55
  HARV (DB, 2023): GVF-GCC div=5.39  GVF-NDVI div=1.44  GCC-NDVI div=6.26
  KONA (AG, 2023): GVF-GCC div=1.88  GVF-NDVI div=2.19  GCC-NDVI div=0.47
  ORNL (DB, 2023): GVF-GCC div=2.89  GVF-NDVI div=2.59  GCC-NDVI div=0.81
  DELA (DB, 2023): GVF-GCC div=2.58  GVF-NDVI div=1.70  GCC-NDVI div=2.47
  TALL (EN, 2023): GVF-GCC div=2.68  GVF-NDVI div=4.10  GCC-NDVI div=1.48
  CPER (GR, 2023): GVF-GCC div=2.26  GVF-NDVI div=2.13  GCC-NDVI div=0.66
  STER (AG, 2023): GVF-GCC div=n/a  GVF-NDVI div=8.89  GCC-NDVI div=n/a
  MOAB (GR, 2023): GVF-GCC div=3.13  GVF-NDVI div=3.48  GCC-NDVI div=1.29
  JORN (GR, 2023): GVF-GCC div=5.13  GVF-NDVI div=6.02  GCC-NDVI div=7.97
  SRER (SH, 2023): GVF-GCC div=0.65  GVF-NDVI div=1.62  GCC-NDVI div=1.13
  ONAQ (SH, 2023): GVF-GCC div=1.99  GVF-NDVI div=2.70  GCC-NDVI div=4.45

Wrote 12/12 rows to /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomal

,site,roi,veg,year,gvf_sos,gvf_mos,gvf_dos,gvf_eos,gcc_sos,gcc_mos,...,ndvi_eos,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,HARV,NEON.D01.HARV.DP1.00033_DB_1000,DB,2023,101.0,172.0,221.0,334.0,2.0,2.0,...,321.0,5.391176,74.75,0.051890,1.435059,19.25,0.060059,6.263044,87.00,0.048759
1,BART,NEON.D01.BART.DP1.00033_DB_1000,DB,2023,101.0,172.0,221.0,334.0,125.0,133.0,...,308.0,1.749500,24.00,0.035214,1.442935,19.75,0.032221,1.547836,21.25,0.029979
2,SRER,NEON.D14.SRER.DP1.00033_SH_1000,SH,2023,1.0,244.0,250.0,298.0,2.0,239.0,...,343.0,0.653057,8.00,0.081629,1.624471,21.50,0.088757,1.129437,15.00,0.058008
3,DELA,NEON.D08.DELA.DP1.00033_DB_1000,DB,2023,51.0,138.0,211.0,350.0,63.0,95.0,...,357.0,2.580651,35.75,0.027079,1.702095,23.50,0.023523,2.471499,34.25,0.025071
4,CPER,NEON.D10.CPER.DP1.00033_GR_1000,GR,2023,121.0,188.0,194.0,313.0,115.0,150.0,...,282.0,2.263203,31.50,0.013203,2.127256,29.25,0.037971,0.661021,8.75,0.036021


In [10]:
cols = [
    "site", "veg", "year",
    "gvf_sos", "gcc_sos", "ndvi_sos",
    "gvf_vs_ndvi_div", "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    "gvf_vs_gcc_div", "gcc_vs_ndvi_div",
]
cols = [c for c in cols if c in df.columns]
display(top_n(df, by=SORT, n=TOP)[cols])
display(group_summary(df, by="veg"))


,site,veg,year,gvf_sos,gcc_sos,ndvi_sos,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gvf_vs_gcc_div,gcc_vs_ndvi_div
0,HARV,DB,2023,101.0,2.0,106.0,1.435059,19.25,0.060059,5.391176,6.263044
1,BART,DB,2023,101.0,125.0,104.0,1.442935,19.75,0.032221,1.749500,1.547836
2,SRER,SH,2023,1.0,2.0,32.0,1.624471,21.50,0.088757,0.653057,1.129437
3,DELA,DB,2023,51.0,63.0,60.0,1.702095,23.50,0.023523,2.580651,2.471499
4,CPER,GR,2023,121.0,115.0,105.0,2.127256,29.25,0.037971,2.263203,0.661021
5,KONA,AG,2023,94.0,163.0,159.0,2.189103,30.25,0.028389,1.880286,0.472121
6,ORNL,DB,2023,68.0,84.0,85.0,2.588200,32.50,0.266771,2.893593,0.812527
7,ONAQ,SH,2023,85.0,79.0,2.0,2.701735,36.50,0.094593,1.987758,4.454176
8,MOAB,GR,2023,1.0,2.0,12.0,3.483270,47.75,0.072555,3.125984,1.287827
9,TALL,EN,2023,76.0,31.0,2.0,4.098530,56.25,0.080673,2.681390,1.476684


,veg,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,AG,1.880286,26.000,0.066156,5.538550,76.375000,0.083192,0.472121,6.250,0.057740
1,DB,3.153730,43.625,0.037658,1.792072,23.750000,0.095644,2.773727,37.625,0.086227
2,EN,2.681390,36.750,0.056390,4.098530,56.250000,0.080673,1.476684,19.500,0.083827
3,GR,3.507660,47.750,0.096946,3.876714,53.166667,0.079095,3.307046,45.250,0.074903
4,SH,1.320408,17.500,0.070408,2.163103,29.000000,0.091675,2.791806,37.750,0.095378


## Satellite Data Gap boxplot by veg

Distribution of `gvf_vs_ndvi_gap` by vegetation type. Spin-up sites are red
diamonds so we can see how much they inflate the apparent discrepancy
(especially EN / GR; DB barely moves).

True lag / compression examples and the DB-vs-shrub/mixed effect-size test are
in the section after golden ranking (same clean, non-spin-up pool).

Writes `anomaly_pipeline/output/boxplot/<FOLDER>_BOXPLOT.png`.


In [11]:
boxplot_path = plot_gap_boxplot_by_veg(csv_path, ANOMALY_DIR)
print(boxplot_path)

Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png


## Golden standard ranking

Drop spin-up, then rank sites by combined GVF-GCC / GVF-NDVI divergence (gap +
DTW). Closed-canopy **DB** sites are flagged as the control group: most uniform
at VIIRS scales, tightest cross-product agreement, almost no spin-up. Their
gap/DTW distribution is the irreducible baseline under ideal conditions.

Top ranks ≈ small disagreement (baseline); mid/lower ranks still include
larger offsets. The next section pulls lag/compression examples from metadata
and tests whether shrub/mixed gap exceeds the DB baseline (Cohen's d).

Needs scores under `output/metadata/`. Writes `output/golden_standard_ranking.csv`.


In [12]:
rank_path = build_golden_ranking(ANOMALY_DIR)
rank = load_table(rank_path)
rank_cols = [
    "rank", "site", "veg", "year", "source", "golden_candidate",
    "combined_div", "combined_gap", "combined_dtw",
]
rank_cols = [c for c in rank_cols if c in rank.columns]
print(rank_path, "|", len(rank), "rows |", int(rank["golden_candidate"].sum()), "DB candidates")
display(rank.head(15)[rank_cols])
display(rank.loc[rank["golden_candidate"]].head(15)[rank_cols])

Ranked 43 site-years (22 DB golden candidates); excluded spin-up=True
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv | 43 rows | 22 DB candidates


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
1,2,SRER,SH,2024,GBOV_2024,False,0.708393,9.125,0.056607
2,3,robinson2,DB,2023,GoldenSites_2023,True,0.919424,12.500,0.026567
3,4,HARV,DB,2024,GBOV_2024,True,1.019032,13.750,0.036889
4,5,bigtraillake,EN,2023,GoldenSites_2023,False,1.039068,13.750,0.056925
5,6,willowcreek,DB,2023,GoldenSites_2023,True,1.039389,13.625,0.066175
6,7,BART,DB,2024,GBOV_2024,True,1.126926,15.250,0.037641
7,8,morganmonroe2,DB,2023,GoldenSites_2023,True,1.143439,15.625,0.027367
8,9,dukehw,DB,2023,GoldenSites_2023,True,1.252109,17.125,0.028894
9,10,arkansaswhitaker,AG,2023,GoldenSites_2023,False,1.342080,18.250,0.038509


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
2,3,robinson2,DB,2023,GoldenSites_2023,True,0.919424,12.500,0.026567
3,4,HARV,DB,2024,GBOV_2024,True,1.019032,13.750,0.036889
5,6,willowcreek,DB,2023,GoldenSites_2023,True,1.039389,13.625,0.066175
6,7,BART,DB,2024,GBOV_2024,True,1.126926,15.250,0.037641
7,8,morganmonroe2,DB,2023,GoldenSites_2023,True,1.143439,15.625,0.027367
8,9,dukehw,DB,2023,GoldenSites_2023,True,1.252109,17.125,0.028894
10,11,DELA,DB,2024,GBOV_2024,True,1.524166,20.875,0.033095
12,13,BART,DB,2023,GBOV_2023,True,1.596217,21.875,0.033717
13,14,SCBI,DB,2023,GoldenSites_2023,True,1.657652,22.875,0.023724


## Lag, compression, and effect size vs DB baseline

Same clean pool as ranking (spin-up excluded). Two related questions in one pass:

1. **Examples from the CSVs** — how we spot lag/compression in
   `metadata/` (and why they are not at the top of
   `golden_standard_ranking.csv`):
   - **Lag-ish:** large `|gvf_sos − ndvi_sos|` with a plausible green-up span
     (not spin-up).
      - ~0–15 typical
      - 15–40 worth a look
      - 40+ lag candidate.
   - **Compression-ish:** green-up length ratio
     `(gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)` 
      - ratio = 1 -> GVF's green-up phase took exactly as many days as GCC's. No stretching, no squeezing
      - ratio < 1 -> the numerator (GVF's duration) is smaller than the denominator (GCC's duration). GVF's green-up happened in fewer days than GCC's and GVF is compressed relative to GCC.
      - ratio > 1 ->  GVF's duration is bigger. GVF took longer to go from onset to peak than GCC did, GVF is stretched relative to GCC.

2. **Effect-size test** — is shrub/mixed (`SH`+`GR`+`EN`) `gvf_vs_ndvi_gap`
   larger than the DB golden-standard mean? Cohen's d + one-sided Welch t-test
   (`H1: mixed > DB`). Only *excess* beyond the DB baseline supports a
   land-cover-driven lag claim.

Caveat: small `n` for shrub/mixed means the test can be underpowered; treat a
non-significant result as "not yet demonstrated," not proof of no effect.


In [13]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

from shared.data_collection import load_all_scores, load_table

# --- combined metadata (same sources ranking uses) ---
all_scores = load_all_scores(ANOMALY_DIR)
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0)
all_scores["sos_lag_gvf_ndvi"] = all_scores["gvf_sos"] - all_scores["ndvi_sos"]
all_scores["gvf_greenup"] = all_scores["gvf_mos"] - all_scores["gvf_sos"]
all_scores["gcc_greenup"] = all_scores["gcc_mos"] - all_scores["gcc_sos"]
all_scores["comp_gvf_gcc"] = all_scores["gvf_greenup"] / all_scores["gcc_greenup"].replace(0, np.nan)

clean = all_scores.loc[~all_scores["spin_up"]].copy()
print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"clean={len(clean)} | sources={sorted(all_scores['source'].unique())}"
)

example_cols = [
    "site", "veg", "year", "source",
    "gvf_sos", "ndvi_sos", "sos_lag_gvf_ndvi",
    "gvf_greenup", "gcc_greenup", "comp_gvf_gcc",
    "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
]

gs = clean.loc[clean["source"].eq("GoldenSites_2023")].copy()
print(f"\nGoldenSites_2023 clean n={len(gs)}")

print("\nLag-ish examples (largest |GVF SOS − NDVI SOS|, non-spin-up):")
display(
    gs.assign(abs_lag=gs["sos_lag_gvf_ndvi"].abs())
    .sort_values("abs_lag", ascending=False)
    .head(8)[example_cols]
)

print("Compression-ish examples (green-up ratio farthest from 1):")
display(
    gs.assign(ratio_dev=(gs["comp_gvf_gcc"] - 1).abs())
    .sort_values("ratio_dev", ascending=False)
    .head(8)[example_cols]
)

rank = load_table(ANOMALY_DIR / "golden_standard_ranking.csv")
print("\nTop 8 golden ranks (baseline = small combined_div):")
display(rank.head(8)[[c for c in [
    "rank", "site", "veg", "source", "combined_div", "combined_gap",
    "combined_dtw", "golden_candidate",
] if c in rank.columns]])

print("\nMid/lower GoldenSites rows in ranking (still can show lag/compression):")
gs_rank = rank.loc[rank["source"].eq("GoldenSites_2023")]
display(gs_rank.tail(8)[[c for c in [
    "rank", "site", "veg", "combined_div", "combined_gap", "gvf_vs_ndvi_gap",
] if c in gs_rank.columns]])

# --- effect size: DB baseline vs shrub/mixed (SH+GR+EN) ---
db_gap = clean.loc[clean["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
mixed_gap = clean.loc[clean["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

def cohens_d(a: pd.Series, b: pd.Series) -> float:
    """Cohen's d for (b - a) / pooled SD (a=DB control, b=shrub/mixed)."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return float("nan")
    var_p = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    return (b.mean() - a.mean()) / np.sqrt(var_p)

d = cohens_d(db_gap, mixed_gap)
tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

print("\nEffect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)")
print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
if tt.pvalue >= 0.05:
    print(
        "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
        "detectable excess lag over DB (underpowered if n_mixed is small)."
    )
else:
    print(
        "  -> significant excess gap in shrub/mixed beyond the DB baseline "
        "(still check n and spin-up screening before claiming ecology)."
    )


rows=50 | spin-up=7 | clean=43 | sources=['GBOV_2023', 'GBOV_2024', 'GoldenSites_2023']

GoldenSites_2023 clean n=22

Lag-ish examples (largest |GVF SOS − NDVI SOS|, non-spin-up):


,site,veg,year,source,gvf_sos,ndvi_sos,sos_lag_gvf_ndvi,gvf_greenup,gcc_greenup,comp_gvf_gcc,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
39,usgseros,DB,2023,GoldenSites_2023,90.0,3.0,87.0,109.0,30.0,3.633333,37.50,0.069924
44,turkeypointdbf,DB,2023,GoldenSites_2023,97.0,12.0,85.0,59.0,14.0,4.214286,51.75,0.040612
33,mandani2,AG,2023,GoldenSites_2023,103.0,155.0,-52.0,89.0,34.0,2.617647,23.75,0.030510
40,ecb4,AG,2023,GoldenSites_2023,49.0,2.0,47.0,148.0,32.0,4.625000,37.50,0.091216
36,ninemileprairie,DB,2023,GoldenSites_2023,101.0,60.0,41.0,109.0,25.0,4.360000,26.75,0.045599
32,coweeta,DB,2023,GoldenSites_2023,68.0,94.0,-26.0,91.0,25.0,3.640000,22.25,0.041029
35,uiefmiscanthus2,AG,2023,GoldenSites_2023,94.0,112.0,-18.0,114.0,32.0,3.562500,26.25,0.025955
30,arkansaswhitaker,AG,2023,GoldenSites_2023,131.0,114.0,17.0,44.0,31.0,1.419355,19.25,0.037298


Compression-ish examples (green-up ratio farthest from 1):


,site,veg,year,source,gvf_sos,ndvi_sos,sos_lag_gvf_ndvi,gvf_greenup,gcc_greenup,comp_gvf_gcc,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
29,cafcookeastltar01,AG,2023,GoldenSites_2023,108.0,112.0,-4.0,70.0,14.0,5.000000,16.25,0.023524
40,ecb4,AG,2023,GoldenSites_2023,49.0,2.0,47.0,148.0,32.0,4.625000,37.50,0.091216
36,ninemileprairie,DB,2023,GoldenSites_2023,101.0,60.0,41.0,109.0,25.0,4.360000,26.75,0.045599
44,turkeypointdbf,DB,2023,GoldenSites_2023,97.0,12.0,85.0,59.0,14.0,4.214286,51.75,0.040612
34,sweetbriar,DB,2023,GoldenSites_2023,74.0,85.0,-11.0,70.0,18.0,3.888889,25.75,0.055022
31,SCBI,DB,2023,GoldenSites_2023,78.0,89.0,-11.0,69.0,18.0,3.833333,19.50,0.027887
32,coweeta,DB,2023,GoldenSites_2023,68.0,94.0,-26.0,91.0,25.0,3.640000,22.25,0.041029
39,usgseros,DB,2023,GoldenSites_2023,90.0,3.0,87.0,109.0,30.0,3.633333,37.50,0.069924



Top 8 golden ranks (baseline = small combined_div):


,rank,site,veg,source,combined_div,combined_gap,combined_dtw,golden_candidate
0,1,blackrockforest,DB,GoldenSites_2023,0.553277,7.250,0.035420,True
1,2,SRER,SH,GBOV_2024,0.708393,9.125,0.056607,False
2,3,robinson2,DB,GoldenSites_2023,0.919424,12.500,0.026567,True
3,4,HARV,DB,GBOV_2024,1.019032,13.750,0.036889,True
4,5,bigtraillake,EN,GoldenSites_2023,1.039068,13.750,0.056925,False
5,6,willowcreek,DB,GoldenSites_2023,1.039389,13.625,0.066175,True
6,7,BART,DB,GBOV_2024,1.126926,15.250,0.037641,True
7,8,morganmonroe2,DB,GoldenSites_2023,1.143439,15.625,0.027367,True



Mid/lower GoldenSites rows in ranking (still can show lag/compression):


,rank,site,veg,combined_div,combined_gap,gvf_vs_ndvi_gap
21,22,coweeta,DB,2.103588,28.875,22.25
25,26,shalehillsczo,DB,2.311405,31.500,31.50
27,28,usgseros,DB,2.366686,32.500,37.50
30,31,turkeypointdbf,DB,3.098430,42.875,51.75
31,32,segawhitepockets,GR,3.374749,45.625,38.50
35,36,arsope3ltar,AG,3.680825,50.625,48.50
36,37,ecb4,AG,4.014764,54.750,37.50
39,40,macleish,DB,4.613742,62.125,59.00



Effect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)
  DB mean gap:      23.5 days (n=22)
  Shrub/mixed mean: 45.6 days (n=12)
  Cohen's d:        1.21  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)
  one-sided p:      0.007  (H1: mixed > DB)
  -> significant excess gap in shrub/mixed beyond the DB baseline (still check n and spin-up screening before claiming ecology).
